In [21]:
import copy

import numpy as np
import pandas as pd
from parse import parse

In [22]:
log_parsing_results = {"initial_energy": None,
                       "initial_geometry": None,
                       "re_optimized_energy": None,
                       "re_optimized_geometry": None,
                       "re_optimization_num_steps": None,
                       "1st_reopt_EQ_energy": None,
                       "1st_reopt_EQ_geometry": None,
                       "2nd_reopt_EQ_energy": None,
                       "2nd_reopt_EQ_geometry": None,
                      }

num_atoms = 12
just_hit_new_itr = False
itr_header_pattern = "# ITR. {iter_num}"
consuming_coordinates = False
start_consuming_coordinates_at = -1 
stop_consuming_coordinates_at = -1 
consuming_energy = False
consume_energy_at = -1
energy_post_coords_pattern = "ENERGY       {energy}                                 ({etcetera}"
saddle_point_found = False
optimized_structure = False
temp_coords = []

# With saddle point optimization AND IRC, we have three stages of the log:
# Stage 1: re-optimization of candidate TS to true saddle point.
# Stage 2: first EQ-finding by IRC.
# Stage 3: second EQ-finding by IRC.

stage = 1 

with open("scratch/C6H6_saddle_validation/C6H6-val_new_model_example.log", "r") as f: 
    for i, line in enumerate(f):
        if line.startswith("# ITR."):
            just_hit_new_itr = True
            iter_parsed = parse(itr_header_pattern, line)
            iter_num = int(iter_parsed["iter_num"])
            if iter_num == 0:
                consuming_energy = True
                consume_energy_at = i + num_atoms + 2
                consuming_coordinates = True
                start_consuming_coordinates_at = i+1
                stop_consuming_coordinates_at = i+num_atoms
        if consuming_coordinates and i >= start_consuming_coordinates_at:
            temp_coord_row = [line.rstrip().split()[0]] + [float(x) for x in line.rstrip().split()[1:]]
            temp_coords.append(temp_coord_row)
            if i == stop_consuming_coordinates_at:
                consuming_coordinates = False
                if stage == 1:
                    if iter_num == 0:
                        log_parsing_results["initial_geometry"] = copy.deepcopy(temp_coords)
                    elif optimized_structure:
                        log_parsing_results["re_optimized_geometry"] = copy.deepcopy(temp_coords)
                elif optimized_structure: 
                    if stage == 2:
                        log_parsing_results["1st_reopt_EQ_geometry"] = copy.deepcopy(temp_coords)
                    elif stage == 3:
                        log_parsing_results["2nd_reopt_EQ_geometry"] = copy.deepcopy(temp_coords)
                    
        if consuming_energy and i == consume_energy_at and iter_num == 0: # Parse initial energy
            log_parsing_results["initial_energy"] = float(line.split()[1])
            consuming_energy = False
            consume_energy_at = -1
        if line.startswith("Optimized structure"):          
            consuming_energy = True
            consume_energy_at = i + num_atoms + 1
            consuming_coordinates = True
            start_consuming_coordinates_at = i+1
            stop_consuming_coordinates_at = i+num_atoms
        if line.startswith("1st-Order Saddle point was found"):
            saddle_point_found = True
            print(f"Saddle point was found after {iter_num} iterations.")

Saddle point was found after 37 iterations.
